In [1]:
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
import numpy as np
from gensim.models import Word2Vec
import fasttext.util
import fasttext
import fasttext.util
import gzip
import os
import pandas as pd
import anthropic
from typing import List, Dict, Any
import json
import asyncio
import re
from tqdm.asyncio import tqdm_asyncio
import nest_asyncio
from datetime import datetime
import sys
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # FutureWarning 제거

In [27]:
pd.set_option('display.max_columns', None)

In [2]:
df = pd.read_excel('../data/centum_data/21.11-24.6환자 CC_PI_치료계획.xlsx')
df['날짜'] = pd.to_datetime(df['날짜'])
df = df.iloc[:,1:]

api = pd.read_csv('../data/info.csv')
api_key = api.loc[0][1]

In [5]:
df = df[['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견']]

In [14]:
df.sample(10).PI

8745                                   * #47 씹으실때 통증, MOB+
24719                           우측 구치부 백반증전체적으로 금가 있고 갈려있음
8910                                                   NaN
13521    * 턱 떨림 있음* both) 3 역할 못함12345678 12345678Dr.남윤...
6637                                            치아마모,(구치부)
26549                                              * 반대 교합
22435                                                  NaN
1950                                                *#22변색
9682                                 * both) 3 교모심함* CA 많음
17636                                                  NaN
Name: PI, dtype: object

In [48]:
import os
import json
import logging
from datetime import datetime
from typing import List, Dict, Optional
import asyncio
import pandas as pd
import anthropic
from tenacity import retry, stop_after_attempt, wait_exponential
import re
from tqdm.asyncio import tqdm as tqdm_asyncio
import nest_asyncio
nest_asyncio.apply()  # Apply the patch for nested event loops


# Disable SettingWithCopyWarning
pd.options.mode.chained_assignment = None
# processed_df = asyncio.run(process_medical_data(df_sample, api_key))
# Configuration class for better organization
class Config:
    MODEL_NAME = "claude-3-5-haiku-20241022"
    MAX_TOKENS = 4096
    TEMPERATURE = 0
    BATCH_SIZE = 50
    SEMAPHORE_LIMIT = 5
    MAX_RETRIES = 3
    CHECKPOINT_DIR = "checkpoints"
    LOG_FILE = "medical_classifier.log"

# Improved logging setup
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(Config.LOG_FILE),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class CheckpointManager:
    """Checkpoint management class"""
    def __init__(self, checkpoint_dir: str = Config.CHECKPOINT_DIR):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(self.checkpoint_dir, exist_ok=True)

    def get_checkpoint_path(self, column: str) -> str:
        return os.path.join(self.checkpoint_dir, f"{column}_checkpoint.parquet")

    def save_checkpoint(self, df: pd.DataFrame, column: str) -> None:
        try:
            df.to_parquet(self.get_checkpoint_path(column))
            logger.info(f"Checkpoint saved for column {column}")
        except Exception as e:
            logger.error(f"Failed to save checkpoint for {column}: {str(e)}")

    def load_checkpoint(self, column: str) -> Optional[pd.DataFrame]:
        path = self.get_checkpoint_path(column)
        if os.path.exists(path):
            try:
                return pd.read_parquet(path)
            except Exception as e:
                logger.error(f"Failed to load checkpoint for {column}: {str(e)}")
        return None

class MedicalTextClassifier:
    def __init__(self, api_key: str):
        self.client = anthropic.Anthropic(api_key=api_key)
        self.semaphore = asyncio.Semaphore(Config.SEMAPHORE_LIMIT)
        self.checkpoint = CheckpointManager()
        self.classifiers = {
            'CC': self._classify_cc,
            '약': self._classify_medication,
            '장치': self._classify_device,
            '습관': self._classify_habit,
            '찜질': self._classify_hot_pack,
            '마사지, 스트레칭': self._classify_massage,
            'PI': self._classify_present_illness
        }

    async def process_all_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """Process all columns with improved error handling and checkpointing"""
        for column in self.classifiers.keys():
            if column in df.columns:
                logger.info(f"Processing column: {column}")
                df = await self._process_column_with_checkpoint(df, column)
        return df

    async def _process_column_with_checkpoint(self, df: pd.DataFrame, column: str) -> pd.DataFrame:
        """Process column with checkpoint support"""
        try:
            # Check for existing checkpoint
            checkpoint_df = self.checkpoint.load_checkpoint(column)
            if checkpoint_df is not None:
                df.update(checkpoint_df)
                logger.info(f"Resumed from checkpoint for {column}")
                return df

            # Process valid texts
            mask = df[column].notna() & df[column].str.strip().astype(bool)
            if not mask.any():
                return df

            texts_with_idx = [(idx, text) for idx, text in df.loc[mask, column].items()]
            dates_with_idx = [(idx, text) for idx, text in df.loc[mask, '날짜'].items()]

            print(texts_with_idx)
            print(dates_with_idx)

            results = await self._safe_process_batches(
                texts=[text for _, text in texts_with_idx],
                original_indices=[idx for idx, _ in texts_with_idx],
                classifier=self.classifiers[column],
                column=column
            )

            if results:
                result_df = pd.DataFrame(results).set_index('index')
                for col in result_df.columns:
                    new_col = f"{column}_{col}"
                    df[new_col] = result_df[col]

            self._cleanup_checkpoint(column)
            return df

        except Exception as e:
            logger.error(f"Critical error processing {column}: {str(e)}")
            raise

    async def _safe_process_batches(self, texts: List[str], original_indices: List[int],
                                  classifier, column: str) -> List[Dict]:
        """Process batches safely with retries and checkpointing"""
        results = []
        batch_size = Config.BATCH_SIZE

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            batch_indices = original_indices[i:i+batch_size]

            try:
                batch_results = await self._process_with_retry(
                    classifier, batch_texts, batch_indices
                )
                results.extend(batch_results)

                # Save partial results
                partial_df = pd.DataFrame(batch_results).set_index('index')
                self.checkpoint.save_checkpoint(partial_df, column)

            except Exception as e:
                logger.error(f"Batch {i//batch_size} failed: {str(e)}")
                continue

        return results

    @retry(stop=stop_after_attempt(Config.MAX_RETRIES),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _process_with_retry(self, classifier, batch_texts: List[str],
                                batch_indices: List[int]) -> List[Dict]:
        """Process with retry logic"""
        async with self.semaphore:
            results = await classifier(batch_texts, self.semaphore)
            return [{"index": idx, **res} for idx, res in zip(batch_indices, results)]

    def _cleanup_checkpoint(self, column: str) -> None:
        """Clean up checkpoint after successful processing"""
        try:
            checkpoint_path = self.checkpoint.get_checkpoint_path(column)
            if os.path.exists(checkpoint_path):
                os.remove(checkpoint_path)
                logger.info(f"Checkpoint cleaned up for {column}")
        except Exception as e:
            logger.error(f"Failed to cleanup checkpoint for {column}: {str(e)}")

    @retry(stop=stop_after_attempt(Config.MAX_RETRIES),
           wait=wait_exponential(multiplier=1, min=2, max=10))
    async def _make_api_call(self, prompt: str, semaphore: asyncio.Semaphore) -> List[Dict]:
        """Improved API call with better error handling"""
        try:
            async with semaphore:
                response = await asyncio.to_thread(
                    self.client.messages.create,
                    model=Config.MODEL_NAME,
                    max_tokens=Config.MAX_TOKENS,
                    temperature=Config.TEMPERATURE,
                    system="JSON 형식으로 응답하세요.",
                    messages=[{"role": "user", "content": prompt}]
                )

                content = response.content[0].text
                logger.debug(f"API Response: {content[:200]}...")

                result = self._validate_and_parse_json(content)
                if not result:
                    raise ValueError("Invalid JSON structure")
                return result

        except Exception as e:
            logger.error(f"API call failed: {str(e)}")
            raise

    def _validate_and_parse_json(self, content: str) -> List[Dict]:
        """Validate and parse JSON response"""
        try:
            # Extract JSON array using regex for more robust parsing
            array_pattern = r'\[(?:[^[\]]*|\[(?:[^[\]]*|\[[^[\]]*\])*\])*\]'
            matches = list(re.finditer(array_pattern, content))

            if not matches:
                return []

            longest_match = max(matches, key=lambda match: len(match.group()))
            potential_json = longest_match.group()

            parsed = json.loads(potential_json)
            if isinstance(parsed, list):
                # Iterate through each item in the list
                for item in parsed:
                    # Remove finish_reason key if present
                    item.pop('finish_reason', None)
                    # Recursively process nested objects
                    for key, value in item.items():
                        if isinstance(value, dict):  # Check if the value is a dictionary
                            value.pop('finish_reason', None)  # Remove key from nested object

                return parsed

            return []  # Return empty list if parsing fails

        except json.JSONDecodeError:
            logger.error(f"JSON parsing failed. Response content: {content[:500]}")
            return []

    # Original classifier methods remain the same but with improved error handling
    async def _classify_cc(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """Classify Chief Complaints"""
        prompt = f"""
            환자의 증상을 설문한 정보입니다. 당신은 구강내과 전문의이며, CC 설문 조사를 통해 결과적으로 입이 안벌어지거나, 나쁜 소리, 통증 등을 파악하여 환자를 분류하길 원합니다.
            다음 주요 증상(CC) 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. location: 통증/증상 위치 (문자열)
            2. pain_type: 통증/증상 종류 (문자열)
            3. painUncomp_desc_jaw: 턱 통증/불편감 턱 관절의 통증, 소리, 움직임 제한 등과 관련된 증상을 포함하는 카테고리 (문자열)
            4. disable_desc_jaw: 턱 관절의 비정상적인 움직임, 소리, 제한된 개구 등의 증상을 다루는 카테고리 (문자열)
            5. muscle_joint_desc_stress: 스트레스로 인한 턱 근육의 긴장, 통증, 이갈이 등의 증상을 포함하는 카테고리 (문자열)
            6. dentalHistory_desc : 교정 치료, 보톡스, 물리치료 등 치과적 개입과 관련된 증상 및 경과를 다루는 카테고리 (문자열)
            7. factor_habbit : 음식 섭취 습관, 수면 자세, 이 악물기 등 일상생활과 연관된 턱 관절 증상을 포함하는 카테고리 (문자열)
            8. severity: 통증/증상 강도 (1-5, 없으면 null)
            9. duration: 지속 기간 (명시된 경우만, 문자열)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "location": "위치",
                "pain_type": "통증 종류",
                "painUncomp_desc_jaw": "환자의 턱 통증/불편감",
                "disable_desc_jaw": "환자의 턱 관절 비정상적 움직임",
                "muscle_joint_desc_stress": "환자의 스트레스로 인한 턱 근육 긴장",
                "dentalHistory_desc": "환자의 교정 치료, 보톡스, 물리치료 등",
                "factor_habbit": "환자의 음식 섭취 습관, 수면 자세, 이 악물기 등",
                "severity": "숫자 또는 null",
                "duration": "기간 또는 null"
            }}]"""
        return await self._make_api_call(prompt, semaphore)

    async def _classify_medication(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """약물 복용 분류"""
        prompt = f"""다음 약물 복용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. medication_type: 약물 종류 (진통제/소염제/근이완제 등)
            2. frequency: 복용 빈도 ('regular': 정기적, 'occasional': 간헐적, 'none': 미복용)
            3. duration: 복용 기간 (명시된 경우만)
            4. compliance: 복약 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "medication_type": "약물 종류",
                "frequency": "복용 빈도",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_device(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """장치 사용 분류"""
        prompt = f"""다음 장치 사용 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. device_type: 장치 종류
            2. usage_pattern: 사용 패턴 ('constant': 상시착용, 'partial': 부분착용, 'rare': 거의미착용)
            3. duration: 사용 기간
            4. compliance: 착용 순응도 ('good': 양호, 'fair': 보통, 'poor': 불량)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "device_type": "장치 종류",
                "usage_pattern": "사용 패턴",
                "duration": "기간 또는 null",
                "compliance": "순응도"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_habit(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """습관 분류"""
        prompt = f"""다음 습관 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. habit_type: 습관 종류 (이갈이/편측성저작 등)
            2. frequency: 빈도 ('high': 매일/자주, 'medium': 가끔, 'low': 거의없음)
            3. awareness: 인지여부 ('aware': 인지, 'unaware': 미인지)
            4. improvement: 개선여부 ('improved': 개선, 'unchanged': 유지, 'worsened': 악화)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "habit_type": "습관 종류",
                "frequency": "빈도",
                "awareness": "인지여부",
                "improvement": "개선여부"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_hot_pack(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """찜질 분류"""
        prompt = f"""다음 찜질 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. status: 찜질 시행 여부 (0: 미시행, 1: 시행)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 찜질 방법 ('hot': 온찜질, 'cold': 냉찜질, 'both': 둘 다, null: 불명확)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "status": 0 또는 1,
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_massage(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """마사지/스트레칭 분류"""
        prompt = f"""다음 마사지/스트레칭 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.

            각 텍스트에 대해 다음 정보를 추출해주세요:
            1. type: 종류 ('massage': 마사지, 'stretching': 스트레칭, 'both': 둘다)
            2. frequency: 시행 빈도 ('high': 매일/자주, 'medium': 주 2-3회, 'low': 주 1회 이하)
            3. duration: 시행 시간 (분 단위 정수, 명시되지 않은 경우 null)
            4. method: 방법 ('self': 자가, 'professional': 전문가, 'both': 둘다)

            텍스트 목록:
            {texts}

            다음 JSON 형식으로 응답해주세요:
            [{{
                "type": "종류",
                "frequency": "빈도",
                "duration": 숫자 또는 null,
                "method": "방법"
            }}]"""

        return await self._make_api_call(prompt, semaphore)

    async def _classify_present_illness(self, texts: List[str], semaphore: asyncio.Semaphore) -> List[Dict]:
        """현재 질환(PI) 분류 - 수정된 프롬프트"""
        prompt = f"""다음 현재 질환(PI) 관련 텍스트들을 분석하여 JSON 형식으로 분류해주세요.
    텍스트 목록:
    {texts}

    각 텍스트에 대해 아래 정보를 추출해주세요:
    - onset: 발현 시기 (예: "3개월 전", "2023년 1월")
    - pattern: 증상 양상 ("constant", "intermittent", "progressive")
    - aggravating_factors: 악화 요인 목록 (리스트 형식)
    - status: 현재 상태 ("improving", "unchanged", "worsening")
    - TMJ_PI_desc: 진단 및 검사 항목 (리스트 형식)
    - TMJ_PI_treatment: 물리치료 항목 (리스트 형식)
    - drug_treatment: 약물치료 항목 (리스트 형식)
    - closing_dentalgear_desc: 교합치료 항목 (리스트 형식)
    - PI_check: 경과관찰 항목 (리스트 형식)

    아래 JSON 형식으로 응답해주세요:
    [{{"onset": "", "pattern": "", "aggravating_factors": [], "status": "", "TMJ_PI_desc": [], "TMJ_PI_treatment": [], "drug_treatment": [], "closing_dentalgear_desc": [], "PI_check": []}}]
    """
        return await self._make_api_call(prompt, semaphore)


async def process_medical_data(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """Process medical data with comprehensive error handling and logging"""
    classifier = MedicalTextClassifier(api_key)
    start_time = datetime.now()
    logger.info(f"Starting medical data processing at {start_time}")

    try:
        # Process data
        processed_df = await classifier.process_all_columns(df)

        # Log statistics
        end_time = datetime.now()
        processing_time = end_time - start_time
        total_rows = len(df)
        processed_columns = [col for col in df.columns if col in classifier.classifiers]

        logger.info("=== Processing Summary ===")
        logger.info(f"Total time: {processing_time}")
        logger.info(f"Total rows processed: {total_rows}")
        logger.info(f"Columns processed: {processed_columns}")

        # Calculate success rates for each column
        for col in processed_columns:
            total_entries = df[col].notna().sum()
            processed_entries = sum(1 for col_name in processed_df.columns
                                 if col_name.startswith(f"{col}_")
                                 and processed_df[col_name].notna().any())
            success_rate = (processed_entries / total_entries * 100) if total_entries > 0 else 0
            logger.info(f"{col} - Success rate: {success_rate:.2f}%")

        return processed_df

    except Exception as e:
        logger.critical(f"Critical error during medical data processing: {str(e)}")
        raise
    finally:
        # Cleanup (without await)
        if classifier.client:  # Check if client is initialized
            classifier.client.close() # Run close() without await
        logger.info("Processing completed and resources cleaned up")

if __name__ == "__main__":
    # Setup logging
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
        handlers=[
            logging.FileHandler(Config.LOG_FILE),
            logging.StreamHandler()
        ]
    )
    logger = logging.getLogger(__name__)

    try:
        # Load sample data
        df_sample = df.sample(10)

        # Set API key (should be in environment variable or config file in production)

        # Process data
        logger.info("Starting sample data processing")
        loop = asyncio.get_event_loop()
        processed_df = loop.run_until_complete(process_medical_data(df_sample, api_key))

        # Save results
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        output_file = f'processed_medical_data_{timestamp}.parquet'
        processed_df.to_parquet(output_file)
        logger.info(f"Data successfully saved to {output_file}")

        # Print basic statistics
        logger.info("\n=== Processing Results ===")
        for column in processed_df.columns:
            if '_' in column:  # Only show derived columns
                valid_count = processed_df[column].notna().sum()
                logger.info(f"{column}: {valid_count} valid entries")

                if processed_df[column].dtype in ['object', 'category']:
                    value_counts = processed_df[column].value_counts()
                    logger.info(f"Value distribution:\n{value_counts}\n")

    except Exception as e:
        logger.error(f"Main execution failed: {str(e)}")
        sys.exit(1)
    finally:
        logger.info("Program execution completed")



2025-02-23 15:02:29,385 - __main__ - INFO - Starting sample data processing
2025-02-23 15:02:29,416 - __main__ - INFO - Starting medical data processing at 2025-02-23 15:02:29.416478
2025-02-23 15:02:29,418 - __main__ - INFO - Processing column: CC


[(10149, '구강내과#2[도착]물리치료 , SS del증상: 턱 통증 소리 전과 동일하게 없어요'), (25651, '인터넷검색, 턱관절, 코골이저는 모르겠는데 주변사람들이 코골이 있다고해요5년전 비염수술하고난 후에 코골이가 생겼어요 오게된 계기가 14살 때 교정을 했었어요 덧니, 돌출입이었는데시간이 지나니까 돌출이 되는거에요 20살때 재교정하고 다시 예쁘게 들어갔는데 시간지나니까 다시 나오더라고요 아래턱 윗턱 얘기하면서 어려서 잘 이해가 안됐고, 또 다른 치과에서 한달 전에 충치검진을 했는데, 교정과 원장님 따로 계셔서 상담을 받아봤어요거기서는 치아의 문제가 아니고 턱의 문제라해서 구강내과 내원했어요오픈바이트라서 재교정하고 싶은데 다시 재발될까봐 해도 되는지, 궁금해요 코골이, 재교정해도되는지, 턱통증해결하고 싶어요 턱관절 인지를 못하고 있다가 얘기 듣고나서 딱딱소리도 나고 피곤하거나  딱딱한 음식 씹으면 통증, VAS 5가만히 있을땐 괜찮아요 질긴음식 (젤리, 떡볶이)자주 먹어요 병원가서 알았는데, 치아가 평평하대요 , 너무 이를 꽉깨물어서 소리가 안날수있다고 들었어요 이를 악무는습관이 있다고 들었어요두통없어요 잠 잘자요 6-7시간정도, 업무 스트레스 많은 편, 해결되는 편이에요보험여부(치아보험(라이나),실손) / 경구피임약'), (8464, '물리치료, SS 상담구강내과#2 -> 빨리 나가보셔야 하셔서 IMP 후 가심'), (16539, '물리치료 , 장치 ck , 근육두께 ck구강내과#3/ 빈증상: 뒷목 어깨 통증은 저번이랑 비슷해요 VAS 3->3       두통없어요       오른쪽 딱소리 가끔 밥먹을때 나긴해요 (크기,빈도수 비슷)        그동안 오른쪽 턱, 귓속 찌릿한 느낌 없었어요        다른 불편감 없었어요       치아끼리 안닿도록 가끔했어요'), (28033, '지인소개, 턱관절- 20대때 양쪽 턱에서 소리 났었는데 치아교합조정 후 없어요1-2주전부터- 무말랭이 먹다가 왼쪽 귀앞 통증 있었어요 VAS7  그후로 씹을때 

2025-02-23 15:02:37,721 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-23 15:02:37,790 - __main__ - INFO - Checkpoint saved for column CC
2025-02-23 15:02:37,796 - __main__ - INFO - Checkpoint cleaned up for CC
2025-02-23 15:02:37,797 - __main__ - INFO - Processing column: 약


[(16539, '약: 다먹었어요/ 불편감 x'), (27204, '약: 이틀치 남았어요. / 복용중 불편감X')]
[(16539, Timestamp('2024-02-02 00:00:00')), (27204, Timestamp('2024-05-21 00:00:00'))]


2025-02-23 15:02:42,726 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-23 15:02:42,733 - __main__ - INFO - Checkpoint saved for column 약
2025-02-23 15:02:42,736 - __main__ - INFO - Checkpoint cleaned up for 약
2025-02-23 15:02:42,737 - __main__ - INFO - Processing column: 습관


...

2025-02-23 15:02:47,848 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-23 15:02:47,856 - __main__ - INFO - Checkpoint saved for column 습관
2025-02-23 15:02:47,860 - __main__ - INFO - Checkpoint cleaned up for 습관
2025-02-23 15:02:47,860 - __main__ - INFO - Processing column: 마사지, 스트레칭


[(16539, '마사지,스트레칭: 딱히 안해요'), (27204, '마사지: 아파서 못하다가 1주일 전부터 다시 했어요.')]
[(16539, Timestamp('2024-02-02 00:00:00')), (27204, Timestamp('2024-05-21 00:00:00'))]


2025-02-23 15:02:53,074 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-23 15:02:53,087 - __main__ - INFO - Checkpoint saved for column 마사지, 스트레칭
2025-02-23 15:02:53,092 - __main__ - INFO - Checkpoint cleaned up for 마사지, 스트레칭
2025-02-23 15:02:53,093 - __main__ - INFO - Processing column: PI


[(10149, '치아 교모 많음교합 변화 가능성 고지장치 2일 착용 후 1일 휴식'), (25651, '치아교모 많음openbite심함'), (8464, '* 치아 마모 있음12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작근의 장애- 측두하악관절자극요법-단순자극측두하악관절자극요법-단순자극 9월 12일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-전기 - K07.66 저작근의 장애- 측두하악관절자극요법-전기자극측두하악관절자극요법-전기자극 9월 12일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-복합 - K07.66 저작근의 장애- 측두하악관절자극요법-복합자극측두하악관절자극요법-복합자극 9월 12일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료SS- SS Splint 인상채득- 숨그린가글물리치료 , SS del [2주후]23-10-13 / SS del, 보증서'), (16539, '12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작근의 장애- 측두하악관절자극요법-단순자극- 분사신장치료측두하악관절자극요법-단순자극 11월 20일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-전기 - K07.66 저작근의 장애- 측두하악관절자극요법-전기자극측두하악관절자극요법-전기자극 11월 20일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료TMJ자극요법-복합 - K07.66 저작근의 장애- 측두하악관절자극요법-복합자극측두하악관절자극요법-복합자극 11월 20일에 측두하악장애 분석검사 시행12345678 12345678Dr.남윤진료동기능적 교합검사 - K07.38 치아위치의 기타 명시된 이상- 동기능적교합검사동기능적교합검사 T-scan 이용하여 동기능적교합검사 실시함- 장치 조정12345678 12345678Dr.남윤진료초음파 - G24.9 상세불명의 근긴

2025-02-23 15:03:06,200 - httpx - INFO - HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"
2025-02-23 15:03:06,215 - __main__ - INFO - Checkpoint saved for column PI
2025-02-23 15:03:06,221 - __main__ - INFO - Checkpoint cleaned up for PI
2025-02-23 15:03:06,222 - __main__ - INFO - === Processing Summary ===
2025-02-23 15:03:06,223 - __main__ - INFO - Total time: 0:00:36.806198
2025-02-23 15:03:06,224 - __main__ - INFO - Total rows processed: 10
2025-02-23 15:03:06,225 - __main__ - INFO - Columns processed: ['CC', '약', '습관', '마사지, 스트레칭', 'PI']
2025-02-23 15:03:06,230 - __main__ - INFO - CC - Success rate: 90.00%
2025-02-23 15:03:06,232 - __main__ - INFO - 약 - Success rate: 150.00%
2025-02-23 15:03:06,233 - __main__ - INFO - 습관 - Success rate: 100.00%
2025-02-23 15:03:06,235 - __main__ - INFO - 마사지, 스트레칭 - Success rate: 150.00%
2025-02-23 15:03:06,236 - __main__ - INFO - PI - Success rate: 112.50%
2025-02-23 15:03:06,238 - __main__ - INFO - Processing completed 

In [39]:
processed_df.columns

Index(['환자번호', '날짜', 'CC', '약', '장치 ', '습관', '찜질 ', '마사지, 스트레칭', 'PI', 'CMO',
       'MMO', 'Cap.pal', 'M.pal', 'Noise', 'Loading', 'Occlusion', 'OJ/OB',
       'Class', 'Midline Shift', 'Deviation', 'CR-CO', 'Tongue ridging',
       'Mucosal ridging', 'Ultrasono', 'Rt', 'Lt', 'End feel', '치료계획',
       'T-scan 악화/개선', 'CBCT 악화/개선', 'CBCT 판독소견', 'CC_location',
       'CC_pain_type', 'CC_painUncomp_desc_jaw', 'CC_disable_desc_jaw',
       'CC_muscle_joint_desc_stress', 'CC_dentalHistory_desc',
       'CC_factor_habbit', 'CC_severity', 'CC_duration', '약_medication_type',
       '약_frequency', '약_duration', '약_compliance', '약_notes', '습관_habit_type',
       '습관_frequency', '습관_awareness', '습관_improvement', '마사지, 스트레칭_type',
       '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration', '마사지, 스트레칭_method',
       'PI_onset', 'PI_pattern', 'PI_aggravating_factors', 'PI_status',
       'PI_TMJ_PI_desc', 'PI_TMJ_PI_treatment', 'PI_drug_treatment',
       'PI_closing_dentalgear_desc', 'PI_PI_check'],
   

In [40]:
processed_df[['환자번호', '날짜',
       'CC_location','CC_pain_type', 'CC_painUncomp_desc_jaw', 'CC_disable_desc_jaw',
       'CC_muscle_joint_desc_stress', 'CC_dentalHistory_desc',
       'CC_factor_habbit', 'CC_severity', 'CC_duration'
        ]]

,환자번호,날짜,CC_location,CC_pain_type,CC_painUncomp_desc_jaw,CC_disable_desc_jaw,CC_muscle_joint_desc_stress,CC_dentalHistory_desc,CC_factor_habbit,CC_severity,CC_duration
8327,2309-48,2024-01-18,"턱, 두부","근육통, 관절통","턱 통증, 두통",소리 없음,None,"물리치료, 장치 체크",None,NaN,None
14905,2211-360,2023-04-21,오른쪽 턱관절,"관절음, 간헐적 통증","소리 감소, 간헐적 통증",관절 소리 감소,None,"물리치료, 장치 체크",딱딱하고 질긴 음식 피하기,3.5,None
5714,2306-195,2023-09-19,"어금니, 귀 앞","교합 불편감, 근육통","교합 변화, 어금니 세게 물림",하품 시 덜그덕거리는 느낌,None,"물리치료, 장치 체크",None,1.0,None
1197,2302-159,2023-03-23,"양쪽 턱, 치아",근육통,"편두통, 감기 증상",입 벌리기 양호,None,"물리치료, 이갈이 장치, 보톡스",질긴 음식 섭취 용이,NaN,보톡스 후 2-3일
8578,2310-173,2024-02-23,턱,관절음,하품 시 턱 소리,턱 빠짐 위험,None,"물리치료, 장치 체크",None,NaN,None
1164,2302-146,2023-05-06,"턱, 입천장","근육통, 건조감","턱 뻐근함, 입 건조함",None,None,"물리치료, 장치 체크, 교합조정",None,3.5,None
24962,2205-86,2022-11-18,왼쪽 턱,관절 불편감,"하품 시 턱 엇갈림, 수면 자세 불편",입 벌리기 제한,None,"물리치료, 장치 체크, 교합조정",옆으로 누워 자기,NaN,None
5838,2306-231,2024-03-04,턱관절,염증성 통증,"두통, 관절 무거움",양쪽 관절 소리,스트레스 많음,보톡스(심미 목적),"수면 중 힘 들어감, 잠 자주 깸",NaN,4일 전 발생
8810,2310-278,2024-02-28,턱,간헐적 통증,식사 중 통증 감소,관절 소리 지속,None,물리치료,"질긴 음식 피하기, 온찜질",NaN,일주일 간 통증 감소
20123,2201-41,2022-01-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
processed_df[['환자번호', '날짜',
       '약_medication_type',
       '약_frequency', '약_duration', '약_compliance', '습관_habit_type',
       '습관_frequency', '습관_awareness', '습관_improvement', '마사지, 스트레칭_type',
       '마사지, 스트레칭_frequency', '마사지, 스트레칭_duration', '마사지, 스트레칭_method'
        ]]

,환자번호,날짜,약_medication_type,약_frequency,약_duration,약_compliance,습관_habit_type,습관_frequency,습관_awareness,습관_improvement,"마사지, 스트레칭_type","마사지, 스트레칭_frequency","마사지, 스트레칭_duration","마사지, 스트레칭_method"
9808,2207-23,2022-10-18,미상,regular,None,fair,편측성저작/이갈이 방지,high,aware,improved,NaN,NaN,NaN,NaN
699,2301-327,2023-04-29,NaN,NaN,NaN,NaN,편측성저작/이갈이 방지,high,aware,improved,NaN,NaN,NaN,NaN
11775,2209-108,2022-10-19,미상,regular,None,good,편측성저작/이갈이 방지,high,aware,improved,NaN,NaN,NaN,NaN
13521,2210-237,2022-11-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14080,2211-154,2023-04-22,NaN,NaN,NaN,NaN,편측성저작/이갈이 방지,medium,unaware,unchanged,NaN,NaN,NaN,NaN
7919,2309-115,2023-09-21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26431,2401-153,2024-04-12,미상,regular,None,good,편측성저작/이갈이 방지,high,aware,improved,both,high,None,self
13696,2210-49,2022-12-16,미상,regular,None,good,편측성저작/이갈이 방지,high,aware,improved,NaN,NaN,NaN,NaN
20722,2202-183,2022-06-03,NaN,NaN,NaN,NaN,편측성저작/이갈이 방지,high,aware,improved,NaN,NaN,NaN,NaN
5780,2306-224,2023-10-10,NaN,NaN,NaN,NaN,편측성저작/이갈이 방지,high,unaware,unchanged,both,medium,None,self


In [43]:
processed_df[[
    'PI_onset',
    'PI_pattern', 'PI_aggravating_factors', 'PI_status', 'PI_TMJ_PI_desc',
    'PI_TMJ_PI_treatment', 'PI_drug_treatment',
    'PI_closing_dentalgear_desc', 'PI_PI_check'
        ]]

,PI_onset,PI_pattern,PI_aggravating_factors,PI_status,PI_TMJ_PI_desc,PI_TMJ_PI_treatment,PI_drug_treatment,PI_closing_dentalgear_desc,PI_PI_check
8327,최근 (정확한 시기 미상),progressive,"[치아 교모, 저작 시 불편감]",unchanged,"[측두하악장애 분석검사 (9월 6일), T-scan 이용 동기능적 교합검사, 근육 ...","[마모물리치료, 측두하악관절 자극요법 (단순/전기/복합 자극), 교근+측두근 보톡스...",[],"[Splint 조정, 동기능적 교합검사, 교정 #4 발치 계획]","[2개월 후 추적 관찰 예정, 근육 상태 재평가]"
14905,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1197,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8578,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1164,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5838,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8810,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
processed_df[['환자번호', '날짜', 'PI', 'PI_onset', 'PI_pattern', 'PI_aggravating_factors', 'PI_status', 'PI_TMJ_PI_desc',
    'PI_TMJ_PI_treatment', 'PI_drug_treatment',
    'PI_closing_dentalgear_desc', 'PI_PI_check'
        ]]

,환자번호,날짜,PI,PI_onset,PI_pattern,PI_aggravating_factors,PI_status,PI_TMJ_PI_desc,PI_TMJ_PI_treatment,PI_drug_treatment,PI_closing_dentalgear_desc,PI_PI_check
10149,2207-318,2022-08-20,치아 교모 많음교합 변화 가능성 고지장치 2일 착용 후 1일 휴식,알 수 없음,intermittent,"[치아 교모, 교합 변화]",unchanged,[],[],[],[가고지장치 2일 착용 후 1일 휴식],[]
25651,2206-265,2022-06-29,치아교모 많음openbite심함,알 수 없음,constant,"[치아교모, openbite]",unchanged,[],[],[],[],[]
8464,2309-98,2023-10-13,* 치아 마모 있음12345678 12345678Dr.남윤진료TMJ자극요법-단순 -...,알 수 없음,intermittent,[],improving,"[측두하악장애 분석검사, SS Splint 인상채득]",[물리치료],[],[SS Splint],[2주 후 확인]
16539,2311-203,2024-02-02,12345678 12345678Dr.남윤진료TMJ자극요법-단순 - K07.66 저작...,알 수 없음,intermittent,[],unchanged,"[측두하악장애 분석검사, 파노라마, CT 촬영, 턱관절의 퇴행성관절염]","[초음파, TMJ 자극요법(단순/전기/복합)]",[],[SS Splint],[2주 후 확인]
28033,2404-99,2024-04-13,"*3교모*4,4 EXT12345678 12345678Dr.박유진진료측두하악장애분석검...",알 수 없음,intermittent,[턱떨림],unchanged,"[측두하악장애 분석검사, 동기능적 교합검사]","[TMJ 자극요법(전기/복합), 교근+측두근 보톡스, 악관절 고착 해소술]","[리보트릴정, 페리슨정, 소론도정]",[],[1주 후 확인]
1043,2301-95,2023-02-23,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2003,2302-47,2023-08-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21660,2203-192,2022-04-18,* 턱떨림 있음12345678 12345678Dr.남윤진료TMJ자극요법-전기 - K...,알 수 없음,constant,[반대교합],unchanged,"[측두하악장애 분석검사, 파노라마, CT 촬영, 턱관절의 퇴행성관절염]","[초음파, 분사신장치료, 악관절 고착 해소술, TMJ 자극요법(단순/전기/복합)]","[페리슨정, 리보트릴정]",[SS Splint],[2주 후 확인]
27204,2402-336,2024-05-21,* #12 반대교합,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8534,2310-126,2023-11-02,"12345678 12345678Dr.남윤진료측두하악장애분석검사, 파노라마, CT촬영...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
